# Metric vs number-of-subjects (eventclf)

Loads the `.mat` training summaries produced by `train_net.py` (saved in `cluster/outputs/`) and visualizes:

1. Best validation accuracy / loss vs the number of subjects used for training (one series per network).
2. Full per-epoch learning curves (train/validation loss & accuracy) for every network and subject count.


In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from scipy.io import loadmat

# ---------------------------------------------------------------------------
# 1. CONFIG
# ---------------------------------------------------------------------------
out_dir = "../cluster/outputs/"

# Fixed ordering / naming for elegant legends
NETS = ["eegnet", "meegnet", "mlp", "vanput"]

# ---------------------------------------------------------------------------
# 2. Load every .mat, extract (net, n_sub) + per-epoch curves + best metrics
# ---------------------------------------------------------------------------
pattern = re.compile(r"^eventclf_(?P<net>eegnet|meegnet|mlp|vanput)_(?P<nsub>\d+)_")

curve_keys = [
    "train_losses",
    "train_accuracies",
    "validation_losses",
    "validation_accuracies",
]
scalar_keys = {
    "validation_accuracy": "best_val_acc",
    "validation_loss": "best_val_loss",
}

records = {}  # records[net][n_sub] = dict of arrays + scalars
for path in sorted(glob.glob(os.path.join(out_dir, "eventclf_*_ALL.mat"))):
    base = os.path.basename(path)
    m = pattern.match(base)
    if m is None:
        continue
    net, n_sub = m.group("net"), int(m.group("nsub"))

    data = loadmat(path)
    entry = {
        "path": path,
        "n_sub": n_sub,
        "epochs": int(np.atleast_1d(data["epoch"]).squeeze()),
    }
    for key in curve_keys:
        entry[key] = np.atleast_1d(data[key]).ravel()
    for key, out in scalar_keys.items():
        entry[out] = float(np.atleast_1d(data[key]).squeeze())

    records.setdefault(net, {})[n_sub] = entry

# All distinct subject counts, sorted ascending (linear x-axis).
all_nsub = sorted({ns for per_net in records.values() for ns in per_net})
# Restrict to nets present in the folder, keep order.
present_nets = [n for n in NETS if n in records]

print("Networks:", present_nets)
print("Subject counts:", all_nsub)

In [ ]:
# --- sanity check: print the parsed summary table ------------------------
print(
    f"{'network':<12}{'n_sub':>6}{'best_val_acc':>14}{'best_val_loss':>14}{'epochs':>8}"
)
for net in present_nets:
    for ns in sorted(records[net]):
        e = records[net][ns]
        print(
            f"{net:<12}{ns:>6}{e['best_val_acc']:>14.4f}{e['best_val_loss']:>14.4f}{e['epochs']:>8}"
        )

In [ ]:
# ---------------------------------------------------------------------------
# Scientific-paper figure styling (matplotlib-only, no seaborn dependency)
# ---------------------------------------------------------------------------
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.constrained_layout.use": True,
    }
)

net_colors = {
    "eegnet": "#1f77b4",
    "meegnet": "#d62728",
    "mlp": "#2ca02c",
    "vanput": "#9467bd",
}

CHANCE = 0.5  # binary event classification chance level.


In [ ]:
# ---------------------------------------------------------------------------
# FIGURE 1: best validation metric vs n_sub (1x2 side-by-side), linear x
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.8))

# Left: validation accuracy
ax = axes[0]
for net in present_nets:
    ns_sorted = sorted(records[net])
    vals = [records[net][ns]["best_val_acc"] for ns in ns_sorted]
    ax.plot(ns_sorted, vals, marker="o", ms=4, lw=1.6, color=net_colors[net], label=net)
ax.axhline(CHANCE, color="0.4", ls="--", lw=1.0, label="chance")
ax.set_xlabel("n subjects (train)")
ax.set_ylabel("Best validation accuracy")
ax.set_title("Validation accuracy")

# Right: validation loss
ax = axes[1]
for net in present_nets:
    xs_sorted = sorted(records[net])
    vals = [records[net][ns]["best_val_loss"] for ns in xs_sorted]
    ax.plot(xs_sorted, vals, marker="o", ms=4, lw=1.6, color=net_colors[net], label=net)
ax.set_xlabel("n subjects (train)")
ax.set_ylabel("Best validation loss")
ax.set_title("Validation loss")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    ncol=5,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
)
fig.savefig("metric_vs_nsub.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# FIGURE 2: 2x2 grid of per-epoch learning curves (x = epoch)
# One metric per panel; a shared n_sub -> color colormap across all panels.
# ---------------------------------------------------------------------------
cmap = plt.get_cmap("viridis")
norm = mcolors.Normalize(vmin=min(all_nsub), vmax=max(all_nsub))

panels = [
    ("train_losses", "Training loss", "loss"),
    ("train_accuracies", "Training accuracy", "accuracy"),
    ("validation_losses", "Validation loss", "loss"),
    ("validation_accuracies", "Validation accuracy", "accuracy"),
]

fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0))
axes = axes.ravel()

for ax, (key, title, kind) in zip(axes, panels):
    for ns in all_nsub:
        for net in present_nets:
            if ns not in records[net]:
                continue
            curve = records[net][ns][key]
            epochs = np.arange(1, len(curve) + 1)
            ax.plot(
                epochs,
                curve,
                color=cmap(norm(ns)),
                lw=1.1,
                label=str(ns) if net == present_nets[0] else None,
            )
    ax.set_xlabel("Epoch")
    ax.set_ylabel(kind.capitalize())
    ax.set_title(title)
    if kind == "accuracy":
        ax.set_ylim(0, 1.02)

# Fig-level colorbar legend for n_sub (deduplicated labels).
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, orientation="horizontal", pad=0.06, aspect=40)
cbar.set_label("n_subj (train)")
cbar.set_ticks(all_nsub)

fig.savefig("learning_curves_2x2.png", bbox_inches="tight")
plt.show()